### [Replacing Classical Forecasting With Deep Learning Transformers](https://pub.towardsai.net/replacing-classical-forecasting-with-deep-learning-transformers-bfc5f874055b)

In [ ]:
import os, sys
import subprocess
import importlib.util

def has_module(name: str) -> bool:
  return importlib.util.find_spec(name) is not None

def install_if_missing(package, import_name=None):
  import_name = import_name or package.split('[')[0]  # handle extras like langchain[google]
  try:
    __import__(import_name.replace('-', '_'))
  except ImportError:
    print(f"Installing {package}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

In [ ]:
to_install = []
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
  install_if_missing('pytorch-lightning', 'pytorch_lightning')
  install_if_missing('pytorch-forecasting', 'pytorch_forecasting')
else:
  if not has_module('numpy'):
    to_install.append('numpy')
  if not has_module('pandas'):
    to_install.append('pandas')
  if not has_module('torch'):
    to_install.append('torch')
  if not has_module('matplotlib'):
    to_install.append('matplotlib')
  if not has_module('seaborn'):
    to_install.append('seaborn')
  if not has_module('pytorch_lightning'):
    to_install.append('pytorch-lightning')
  if not has_module('pytorch_forecasting'):
    to_install.append('pytorch-forecasting')

if to_install:
  print("Installing missing packages:", to_install)
  subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + to_install)
  print("\nRestart the runtime/kernel now, then run the next cell (imports).")

print("✓ All packages ready! All required packages already available.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch

import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss, RMSE, MAE
from pytorch_forecasting.models.temporal_fusion_transformer.tuning import optimize_hyperparameters

# Set random seeds for reproducibility
pl.seed_everything(42)

import warnings
warnings.filterwarnings('ignore')

In [ ]:
def generate_realistic_data(n_samples=15000):
    """Generate realistic multivariate time series data"""
    np.random.seed(42)

    dates = pd.date_range(start='2020-01-01', periods=n_samples, freq='h')
    stores = ['Store_A', 'Store_B', 'Store_C', 'Store_D']
    products = ['Product_1', 'Product_2', 'Product_3']

    data_list = []

    for store in stores:
        for product in products:
            # Base demand level varies by store and product
            base_demand = np.random.uniform(30, 80)

            # Trend component
            trend = np.linspace(0, 20, n_samples) * np.random.uniform(0.5, 1.5)

            # Multiple seasonality patterns
            daily_season = 15 * np.sin(2 * np.pi * np.arange(n_samples) / 24)
            weekly_season = 10 * np.sin(2 * np.pi * np.arange(n_samples) / (24*7))
            yearly_season = 5 * np.sin(2 * np.pi * np.arange(n_samples) / (24*365))

            # Noise
            noise = np.random.normal(0, 5, n_samples)

            # Combine components
            sales = base_demand + trend + daily_season + weekly_season + yearly_season + noise
            sales = np.maximum(sales, 0)  # Ensure non-negative

            # Generate features
            temp_df = pd.DataFrame({
                'date': dates,
                'store_id': store,
                'product_id': product,
                'sales': sales,
                'price': np.random.uniform(15, 45, n_samples) + 5 * np.sin(np.arange(n_samples) / 100),
                'promotion': np.random.choice([0, 1], n_samples, p=[0.85, 0.15]),
                'day_of_week': dates.dayofweek,
                'hour': dates.hour,
                'month': dates.month,
                'is_weekend': (dates.dayofweek >= 5).astype(int),
            })

            data_list.append(temp_df)

    data = pd.concat(data_list, ignore_index=True)

    # Add time index
    data = data.sort_values(['store_id', 'product_id', 'date'])
    data['time_idx'] = (data['date'] - data['date'].min()).dt.total_seconds() // 3600
    data['time_idx'] = data['time_idx'].astype(int)

    # Add lagged features
    data['sales_lag_24'] = data.groupby(['store_id', 'product_id'])['sales'].shift(24)
    data['sales_lag_168'] = data.groupby(['store_id', 'product_id'])['sales'].shift(168)

    # Fill NaN values
    data = data.fillna(method='bfill')

    return data

# Generate data
print("Generating synthetic data...")
data = generate_realistic_data()
print(f"Data shape: {data.shape}")
print(f"Date range: {data['date'].min()} to {data['date'].max()}")

In [ ]:
display(data.sample(15))

In [ ]:
# Define forecasting parameters
max_encoder_length = 168  # Use 7 days (168 hours) of history
max_prediction_length = 24  # Predict 24 hours ahead

training_cutoff = data['time_idx'].max() - max_prediction_length

# Convert categorical features to category dtype
# This is required by PyTorch Forecasting for columns specified as categoricals
data['day_of_week'] = data['day_of_week'].astype(str).astype('category')
data['hour'] = data['hour'].astype(str).astype('category')
data['month'] = data['month'].astype(str).astype('category')
data['is_weekend'] = data['is_weekend'].astype(str).astype('category')
data['promotion'] = data['promotion'].astype(str).astype('category')

# Create training dataset with updated parameters
training = TimeSeriesDataSet(
    data[lambda x: x.time_idx <= training_cutoff],
    time_idx='time_idx',
    target='sales',
    group_ids=['store_id', 'product_id'],
    min_encoder_length=max_encoder_length // 2,
    max_encoder_length=max_encoder_length,
    min_prediction_length=1,
    max_prediction_length=max_prediction_length,

    # Static categoricals (don't change over time)
    static_categoricals=['store_id', 'product_id'],

    # Time-varying known (available at prediction time)
    time_varying_known_categoricals=['day_of_week', 'hour', 'month', 'is_weekend', 'promotion'],
    time_varying_known_reals=['time_idx', 'price'],

    # Time-varying unknown (not available at prediction time)
    time_varying_unknown_reals=['sales', 'sales_lag_24', 'sales_lag_168'],

    # Normalization
    target_normalizer=GroupNormalizer(
        groups=['store_id', 'product_id'],
        transformation='softplus'  # Better for non-negative data
    ),

    # Additional features
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True,
)

# Create validation dataset
validation = TimeSeriesDataSet.from_dataset(
    training,
    data,
    predict=True,
    stop_randomization=True
)

# Create dataloaders with updated parameters
batch_size = 128
train_dataloader = training.to_dataloader(
    train=True,
    batch_size=batch_size,
    num_workers=0,  # Set to 0 for Windows, 4+ for Linux
    persistent_workers=False
)

val_dataloader = validation.to_dataloader(
    train=False,
    batch_size=batch_size,
    num_workers=0,
    persistent_workers=False
)

print(f"\nTraining samples: {len(training)}")
print(f"Validation samples: {len(validation)}")

In [ ]:
# Configure TFT model with latest best practices
tft = TemporalFusionTransformer.from_dataset(
    training,
    # Architecture
    hidden_size=64,  # Increased from 32
    lstm_layers=2,
    attention_head_size=4,
    dropout=0.2,  # Increased regularization
    hidden_continuous_size=16,

    # Output configuration
    output_size=7,  # 7 quantiles for probabilistic forecasting
    loss=QuantileLoss(),

    # Optimizer configuration (updated for PyTorch 2.x)
    learning_rate=3e-3,
    optimizer='ranger',  # Better than Adam for time series

    # Logging
    log_interval=10,
    log_val_interval=1,

    # Learning rate schedule
    reduce_on_plateau_patience=4,
    reduce_on_plateau_reduction=2.0,
)

print(f"\nModel parameters: {tft.size()/1e6:.2f}M")

In [ ]:
# Setup callbacks with latest Lightning 2.x syntax
early_stop_callback = EarlyStopping(
    monitor="val_loss",
    min_delta=1e-4,
    patience=10,
    verbose=True,
    mode="min"
)

checkpoint_callback = ModelCheckpoint(
    monitor='val_loss',
    dirpath='/content/outputs/checkpoints',
    filename='tft-{epoch:02d}-{val_loss:.2f}',
    save_top_k=3,
    mode='min',
)

# Create trainer with PyTorch Lightning 2.x API
trainer = pl.Trainer(
    max_epochs=50,
    accelerator='auto',  # Automatically detect GPU/CPU
    devices=1,
    gradient_clip_val=0.1,
    callbacks=[early_stop_callback, checkpoint_callback],
    enable_progress_bar=True,
    enable_model_summary=True,
    log_every_n_steps=10,
    # New in Lightning 2.x
    precision='32-true',  # Use '16-mixed' for mixed precision on GPU
)

# Train model
print("\nTraining model...")
trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
)

In [ ]:
# Load best model
best_model_path = checkpoint_callback.best_model_path
print(f"\nBest model: {best_model_path}")

best_tft = TemporalFusionTransformer.load_from_checkpoint(best_model_path)

# Generate predictions
print("\nGenerating predictions...")
predictions = best_tft.predict(
    val_dataloader,
    mode="prediction",
    return_x=True,
    return_y=True,
)

# Extract predictions and actuals
actuals = predictions.y[0]  # First batch
forecast = predictions.output[0]  # Predictions
x = predictions.x

# Calculate metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(actuals.cpu().numpy(), forecast.cpu().numpy())
rmse = np.sqrt(mean_squared_error(actuals.cpu().numpy(), forecast.cpu().numpy()))
r2 = r2_score(actuals.cpu().numpy(), forecast.cpu().numpy())

print(f"\nMetrics:")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.4f}")

In [ ]:
# Visualize predictions
def plot_prediction(idx=0, n_samples=5):
    """Plot predictions vs actuals for multiple samples"""
    fig, axes = plt.subplots(n_samples, 1, figsize=(15, 3*n_samples))

    if n_samples == 1:
        axes = [axes]

    for i, ax in enumerate(axes):
        if i >= len(predictions.output):
            break

        actual = predictions.y[i].cpu().numpy()
        pred = predictions.output[i].cpu().numpy()

        # Plot
        time_steps = np.arange(len(actual))
        ax.plot(time_steps, actual, 'o-', label='Actual', linewidth=2, markersize=4)
        ax.plot(time_steps, pred, 's--', label='Predicted', linewidth=2, markersize=4, alpha=0.7)

        # Calculate sample metrics
        sample_mae = mean_absolute_error(actual, pred)
        sample_rmse = np.sqrt(mean_squared_error(actual, pred))

        ax.set_title(f'Sample {i+1} - MAE: {sample_mae:.2f}, RMSE: {sample_rmse:.2f}')
        ax.set_xlabel('Time Step (hours)')
        ax.set_ylabel('Sales')
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('/mnt/user-data/outputs/tft_predictions_latest.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("\nPrediction plot saved!")

plot_prediction(n_samples=5)

In [ ]:
# Feature importance analysis
interpretation = best_tft.interpret_output(
    predictions.x,
    reduction='sum'
)

# Plot attention patterns
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Variable importance
ax = axes[0, 0]
importance = interpretation['encoder_variables'].cpu().numpy()
variables = list(training.reals + training.flat_categoricals)[:len(importance)]
sorted_idx = np.argsort(importance)
ax.barh(np.array(variables)[sorted_idx], importance[sorted_idx])
ax.set_xlabel('Importance')
ax.set_title('Variable Importance')
ax.grid(True, alpha=0.3, axis='x')

# Attention over time
ax = axes[0, 1]
attention = interpretation['attention'].cpu().numpy()[0, 0]  # First sample, first head
im = ax.imshow(attention, aspect='auto', cmap='viridis')
ax.set_xlabel('Encoder Time Steps')
ax.set_ylabel('Decoder Time Steps')
ax.set_title('Attention Weights')
plt.colorbar(im, ax=ax)

# Prediction intervals
ax = axes[1, 0]
sample_idx = 0
actual = predictions.y[sample_idx].cpu().numpy()
quantiles = predictions.output[sample_idx].cpu().numpy() if predictions.output[sample_idx].ndim > 1 else predictions.output[sample_idx].cpu().numpy().reshape(-1, 1)

time_steps = np.arange(len(actual))
ax.plot(time_steps, actual, 'o-', label='Actual', linewidth=2)

if quantiles.shape[1] >= 3:
    ax.plot(time_steps, quantiles[:, 3], '--', label='Median', linewidth=2)
    ax.fill_between(time_steps, quantiles[:, 1], quantiles[:, 5], alpha=0.3, label='50% Interval')
    ax.fill_between(time_steps, quantiles[:, 0], quantiles[:, 6], alpha=0.2, label='90% Interval')

ax.set_xlabel('Time Step')
ax.set_ylabel('Sales')
ax.set_title('Prediction Intervals')
ax.legend()
ax.grid(True, alpha=0.3)

# Residual analysis
ax = axes[1, 1]
all_actuals = []
all_predictions = []
for i in range(min(20, len(predictions.output))):
    all_actuals.extend(predictions.y[i].cpu().numpy())
    all_predictions.extend(predictions.output[i].cpu().numpy() if predictions.output[i].ndim == 1
                          else predictions.output[i].cpu().numpy()[:, 3])  # Use median

residuals = np.array(all_actuals) - np.array(all_predictions)
ax.scatter(all_predictions, residuals, alpha=0.5)
ax.axhline(y=0, color='r', linestyle='--', linewidth=2)
ax.set_xlabel('Predicted Values')
ax.set_ylabel('Residuals')
ax.set_title('Residual Plot')
ax.grid(True, alpha=0.3)

plt.tight_layout()
if not IN_COLAB:
  plt.savefig('/content/tft_analysis_latest.png', dpi=300, bbox_inches='tight')
  print("Analysis plot saved!")
plt.show()
plt.close()

In [ ]:
"""
Modern time series Transformer implementation using PyTorch 2.x features
Includes: Flash Attention, Compiled models, BFloat16 support
"""

import torch
import torch.nn as nn
from torch.nn import functional as F
import math

class ModernTimeSeriesTransformer(nn.Module):
    """
    Transformer with PyTorch 2.x optimizations
    - Flash Attention (if available)
    - Compiled mode support
    - Efficient implementation
    """
    def __init__(
        self,
        d_model=256,
        nhead=8,
        num_encoder_layers=6,
        num_decoder_layers=6,
        dim_feedforward=1024,
        dropout=0.1,
        input_size=1,
        output_size=1,
        max_seq_length=512
    ):
        super().__init__()

        self.d_model = d_model
        self.input_size = input_size
        self.output_size = output_size

        # Input projection
        self.input_projection = nn.Linear(input_size, d_model)

        # Positional encoding using sinusoidal embeddings
        self.register_buffer(
            'pos_encoding',
            self._create_positional_encoding(max_seq_length, d_model)
        )

        # Modern Transformer with native PyTorch 2.x
        # Note: Flash Attention automatically used if available
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,  # Important for modern PyTorch
            norm_first=True,   # Pre-norm architecture (more stable)
        )

        # Output projection
        self.output_projection = nn.Linear(d_model, output_size)

        # Layer normalization
        self.norm = nn.LayerNorm(d_model)

        self._init_weights()

    def _create_positional_encoding(self, max_len, d_model):
        """Create sinusoidal positional encoding"""
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() *
                           (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)

    def _init_weights(self):
        """Initialize weights using modern best practices"""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, src, tgt=None, src_mask=None, tgt_mask=None):
        """
        Args:
            src: Source sequence [batch, src_len, input_size]
            tgt: Target sequence [batch, tgt_len, input_size] (optional)
            src_mask: Source attention mask
            tgt_mask: Target attention mask (causal)
        """
        batch_size, src_len, _ = src.shape

        # Project and add positional encoding
        src = self.input_projection(src)
        src = src + self.pos_encoding[:, :src_len, :]
        src = self.norm(src)

        if tgt is None:
            # Encoder-only mode (for representation learning)
            memory = self.transformer.encoder(src, mask=src_mask)
            output = self.output_projection(memory)
        else:
            # Full encoder-decoder mode
            tgt_len = tgt.shape[1]
            tgt = self.input_projection(tgt)
            tgt = tgt + self.pos_encoding[:, :tgt_len, :]
            tgt = self.norm(tgt)

            # Create causal mask for autoregressive prediction
            if tgt_mask is None:
                tgt_mask = nn.Transformer.generate_square_subsequent_mask(
                    tgt_len, device=tgt.device
                )

            output = self.transformer(
                src, tgt,
                src_mask=src_mask,
                tgt_mask=tgt_mask
            )
            output = self.output_projection(output)

        return output


class TimeSeriesDataModule:
    """Modern data handling for time series"""
    def __init__(self, data, seq_len=96, pred_len=24, batch_size=32):
        self.data = torch.FloatTensor(data).unsqueeze(-1) if data.ndim == 1 else torch.FloatTensor(data)
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.batch_size = batch_size

    def create_sequences(self):
        """Create overlapping sequences"""
        sequences = []
        targets = []

        for i in range(len(self.data) - self.seq_len - self.pred_len):
            seq = self.data[i:i+self.seq_len]
            target = self.data[i+self.seq_len:i+self.seq_len+self.pred_len]
            sequences.append(seq)
            targets.append(target)

        return torch.stack(sequences), torch.stack(targets)

    def get_dataloaders(self, train_split=0.8):
        """Create train/val dataloaders"""
        X, y = self.create_sequences()

        # Split data
        train_size = int(len(X) * train_split)
        X_train, X_val = X[:train_size], X[train_size:]
        y_train, y_val = y[:train_size], y[train_size:]

        # Create datasets
        train_dataset = torch.utils.data.TensorDataset(X_train, y_train)
        val_dataset = torch.utils.data.TensorDataset(X_val, y_val)

        # Create dataloaders with modern settings
        train_loader = torch.utils.data.DataLoader(
            train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=0,
            pin_memory=True if torch.cuda.is_available() else False,
            persistent_workers=False
        )

        val_loader = torch.utils.data.DataLoader(
            val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=0,
            pin_memory=True if torch.cuda.is_available() else False,
            persistent_workers=False
        )

        return train_loader, val_loader


def train_modern_transformer():
    """Training loop with modern PyTorch 2.x features"""

    # Generate synthetic data
    n_points = 10000
    t = np.linspace(0, 100, n_points)
    data = 50 + 10*np.sin(t) + 5*np.sin(5*t) + np.random.normal(0, 2, n_points)

    # Setup data
    dm = TimeSeriesDataModule(data, seq_len=96, pred_len=24, batch_size=64)
    train_loader, val_loader = dm.get_dataloaders()

    # Create model
    model = ModernTimeSeriesTransformer(
        d_model=128,
        nhead=8,
        num_encoder_layers=4,
        num_decoder_layers=4,
        dim_feedforward=512,
        dropout=0.1
    )

    # Move to device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    # Compile model for PyTorch 2.x (significant speedup)
    if hasattr(torch, 'compile'):
        print("Using torch.compile for optimization...")
        model = torch.compile(model, mode='reduce-overhead')

    # Modern optimizer with fused operations
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-4,
        weight_decay=0.01,
        fused=True if torch.cuda.is_available() else False  # Fused AdamW
    )

    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=1e-3,
        epochs=50,
        steps_per_epoch=len(train_loader),
        pct_start=0.3
    )

    # Loss function
    criterion = nn.MSELoss()

    # Training loop with automatic mixed precision (AMP)
    scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

    print(f"Training on {device}")
    print(f"Model parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

    best_val_loss = float('inf')
    train_losses = []
    val_losses = []

    for epoch in range(50):
        # Training
        model.train()
        train_loss = 0

        for src, tgt in train_loader:
            src, tgt = src.to(device), tgt.to(device)

            optimizer.zero_grad(set_to_none=True)  # More efficient than zero_grad()

            # Mixed precision training
            if scaler:
                with torch.cuda.amp.autocast():
                    # Teacher forcing: use target as decoder input
                    output = model(src, tgt)
                    loss = criterion(output, tgt)

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                output = model(src, tgt)
                loss = criterion(output, tgt)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            scheduler.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)
        train_losses.append(train_loss)

        # Validation
        model.eval()
        val_loss = 0

        with torch.no_grad():
            for src, tgt in val_loader:
                src, tgt = src.to(device), tgt.to(device)

                if scaler:
                    with torch.cuda.amp.autocast():
                        output = model(src, tgt)
                        loss = criterion(output, tgt)
                else:
                    output = model(src, tgt)
                    loss = criterion(output, tgt)

                val_loss += loss.item()

        val_loss /= len(val_loader)
        val_losses.append(val_loss)

        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            print(f"New best model found at epoch {epoch} with val loss {val_loss:.4f}")
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
            }, '/content/best_transformer_model.pt')

        if epoch % 5 == 0:
            print(f"Epoch {epoch:3d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

    # Plot training history
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training History')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    if IN_COLAB:
      plt.savefig('/content/training_history_modern.png', dpi=300, bbox_inches='tight')
      print("Training history plot saved!")

    plt.show()
    plt.close()

    return model, train_loader, val_loader

# Train the model
print("Training modern transformer...")
model, train_loader, val_loader = train_modern_transformer()
print("Training completed!")

In [ ]:
"""
Comprehensive comparison using latest libraries
statsmodels 0.14.x, scikit-learn 1.3.x
"""

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from prophet import Prophet  # Facebook Prophet with latest updates
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
import time
import warnings
warnings.filterwarnings('ignore')

def comprehensive_forecasting_comparison():
    """Compare multiple forecasting approaches"""

    # Generate realistic data
    np.random.seed(42)
    n_points = 2000
    dates = pd.date_range(start='2020-01-01', periods=n_points, freq='D')

    trend = np.linspace(100, 200, n_points)
    yearly_season = 20 * np.sin(2 * np.pi * np.arange(n_points) / 365.25)
    weekly_season = 10 * np.sin(2 * np.pi * np.arange(n_points) / 7)
    noise = np.random.normal(0, 5, n_points)

    y = trend + yearly_season + weekly_season + noise

    # Train-test split
    train_size = int(0.85 * n_points)
    train_dates, test_dates = dates[:train_size], dates[train_size:]
    y_train, y_test = y[:train_size], y[train_size:]

    results = {}

    # 1. SARIMAX (Latest statsmodels)
    print("Training SARIMAX...")
    start_time = time.time()
    try:
        sarimax_model = SARIMAX(
            y_train,
            order=(2, 1, 2),
            seasonal_order=(1, 1, 1, 7),
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        sarimax_fit = sarimax_model.fit(disp=False, maxiter=100)
        sarimax_pred = sarimax_fit.forecast(steps=len(y_test))
        sarimax_time = time.time() - start_time

        results['SARIMAX'] = {
            'predictions': sarimax_pred,
            'mae': mean_absolute_error(y_test, sarimax_pred),
            'rmse': np.sqrt(mean_squared_error(y_test, sarimax_pred)),
            'mape': mean_absolute_percentage_error(y_test, sarimax_pred),
            'time': sarimax_time
        }
        print(f"✓ SARIMAX completed in {sarimax_time:.2f}s")
    except Exception as e:
        print(f"✗ SARIMAX failed: {e}")
        results['SARIMAX'] = None

    # 2. Exponential Smoothing (Holt-Winters)
    print("Training Exponential Smoothing...")
    start_time = time.time()
    try:
        es_model = ExponentialSmoothing(
            y_train,
            seasonal_periods=7,
            trend='add',
            seasonal='add',
            initialization_method='estimated'
        )
        es_fit = es_model.fit(optimized=True)
        es_pred = es_fit.forecast(steps=len(y_test))
        es_time = time.time() - start_time

        results['Exp_Smoothing'] = {
            'predictions': es_pred,
            'mae': mean_absolute_error(y_test, es_pred),
            'rmse': np.sqrt(mean_squared_error(y_test, es_pred)),
            'mape': mean_absolute_percentage_error(y_test, es_pred),
            'time': es_time
        }
        print(f"✓ Exp. Smoothing completed in {es_time:.2f}s")
    except Exception as e:
        print(f"✗ Exp. Smoothing failed: {e}")
        results['Exp_Smoothing'] = None

    # 3. Prophet (Facebook)
    print("Training Prophet...")
    start_time = time.time()
    try:
        prophet_df = pd.DataFrame({
            'ds': train_dates,
            'y': y_train
        })

        prophet_model = Prophet(
            yearly_seasonality=True,
            weekly_seasonality=True,
            daily_seasonality=False,
            changepoint_prior_scale=0.05
        )
        prophet_model.fit(prophet_df)

        future = pd.DataFrame({'ds': test_dates})
        prophet_pred = prophet_model.predict(future)['yhat'].values
        prophet_time = time.time() - start_time

        results['Prophet'] = {
            'predictions': prophet_pred,
            'mae': mean_absolute_error(y_test, prophet_pred),
            'rmse': np.sqrt(mean_squared_error(y_test, prophet_pred)),
            'mape': mean_absolute_percentage_error(y_test, prophet_pred),
            'time': prophet_time
        }
        print(f"✓ Prophet completed in {prophet_time:.2f}s")
    except Exception as e:
        print(f"✗ Prophet failed: {e}")
        results['Prophet'] = None

    # 4. Simple LSTM (PyTorch)
    print("Training LSTM...")
    start_time = time.time()

    class SimpleLSTM(nn.Module):
        def __init__(self, input_size=1, hidden_size=64, num_layers=2):
            super().__init__()
            self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
            self.fc = nn.Linear(hidden_size, 1)

        def forward(self, x):
            out, _ = self.lstm(x)
            out = self.fc(out[:, -1, :])
            return out

    # Prepare sequences
    seq_len = 30
    X_lstm, y_lstm = [], []
    for i in range(len(y_train) - seq_len):
        X_lstm.append(y_train[i:i+seq_len])
        y_lstm.append(y_train[i+seq_len])

    X_lstm = torch.FloatTensor(X_lstm).unsqueeze(-1)
    y_lstm = torch.FloatTensor(y_lstm).unsqueeze(-1)

    lstm_model = SimpleLSTM()
    optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.001)
    criterion = nn.MSELoss()

    # Quick training
    lstm_model.train()
    for epoch in range(30):
        optimizer.zero_grad()
        output = lstm_model(X_lstm)
        loss = criterion(output, y_lstm)
        loss.backward()
        optimizer.step()

    # Predict
    lstm_model.eval()
    lstm_preds = []
    current_seq = torch.FloatTensor(y_train[-seq_len:]).unsqueeze(0).unsqueeze(-1)

    with torch.no_grad():
        for _ in range(len(y_test)):
            pred = lstm_model(current_seq)
            lstm_preds.append(pred.item())
            current_seq = torch.cat([current_seq[:, 1:, :], pred.unsqueeze(1)], dim=1)

    lstm_time = time.time() - start_time

    results['LSTM'] = {
        'predictions': np.array(lstm_preds),
        'mae': mean_absolute_error(y_test, lstm_preds),
        'rmse': np.sqrt(mean_squared_error(y_test, lstm_preds)),
        'mape': mean_absolute_percentage_error(y_test, lstm_preds),
        'time': lstm_time
    }
    print(f"✓ LSTM completed in {lstm_time:.2f}s")

    # Visualization
    fig, axes = plt.subplots(3, 2, figsize=(16, 12))

    # Plot forecasts
    ax = axes[0, 0]
    ax.plot(test_dates, y_test, 'k-', label='Actual', linewidth=2)
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
    for i, (name, result) in enumerate(results.items()):
        if result:
            ax.plot(test_dates, result['predictions'], '--',
                   label=name, linewidth=1.5, color=colors[i], alpha=0.8)
    ax.set_title('Forecast Comparison', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylabel('Value')

    # Metrics comparison
    metrics = ['mae', 'rmse', 'mape']
    metric_names = ['MAE', 'RMSE', 'MAPE (%)']

    for idx, (metric, metric_name) in enumerate(zip(metrics, metric_names)):
        if idx < 2:
            ax = axes[0, 1] if idx == 0 else axes[1, 0]
        else:
            ax = axes[1, 1]

        model_names = []
        values = []
        for name, result in results.items():
            if result:
                model_names.append(name)
                val = result[metric] * 100 if metric == 'mape' else result[metric]
                values.append(val)

        ax.bar(model_names, values, color=colors[:len(model_names)], alpha=0.7)
        ax.set_ylabel(metric_name)
        ax.set_title(f'{metric_name} Comparison', fontsize=11, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='y')
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

    # Training time comparison
    ax = axes[2, 0]
    times = [r['time'] for r in results.values() if r]
    names = [n for n, r in results.items() if r]
    ax.bar(names, times, color=colors[:len(names)], alpha=0.7)
    ax.set_ylabel('Time (seconds)')
    ax.set_title('Training Time Comparison', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

    # Summary table
    ax = axes[2, 1]
    ax.axis('tight')
    ax.axis('off')

    table_data = [['Model', 'MAE', 'RMSE', 'MAPE%', 'Time(s)']]
    for name, result in results.items():
        if result:
            table_data.append([
                name,
                f"{result['mae']:.2f}",
                f"{result['rmse']:.2f}",
                f"{result['mape']*100:.2f}",
                f"{result['time']:.2f}"
            ])

    table = ax.table(cellText=table_data, cellLoc='center', loc='center',
                    colWidths=[0.25, 0.15, 0.15, 0.15, 0.15])
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 2)

    for i in range(len(table_data[0])):
        table[(0, i)].set_facecolor('#4472C4')
        table[(0, i)].set_text_props(weight='bold', color='white')

    plt.tight_layout()

    if IN_COLAB:
      plt.savefig('/content/comprehensive_comparison_latest.png', dpi=300, bbox_inches='tight')
      print("Comprehensive comparison plot saved!")

    plt.show()
    plt.close()

    print("\n" + "="*60)
    print("FINAL RESULTS SUMMARY")
    print("="*60)
    for name, result in results.items():
        if result:
            print(f"\n{name}:")
            print(f"  MAE:  {result['mae']:.2f}")
            print(f"  RMSE: {result['rmse']:.2f}")
            print(f"  MAPE: {result['mape']*100:.2f}%")
            print(f"  Time: {result['time']:.2f}s")

    return results

# Run comprehensive comparison
results = comprehensive_forecasting_comparison()